## Налаштування та тренування базових моделей (Part 1 & 2)
Спочатку підготуємо корпус, натренуємо наші моделі Skip-Gram та CBOW, а також завантажимо попередньо натреновану модель GloVe.

In [ ]:
import gensim
from gensim.models import Word2Vec
import gensim.downloader as api
from gensim.utils import simple_preprocess
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)

corpus = """Natural language processing is a fascinating field of artificial intelligence.
Machine learning algorithms can learn patterns from data automatically.
Deep learning uses neural networks with multiple layers.
Word embeddings capture semantic relationships between words.
The quick brown fox jumps over the lazy dog.
Artificial intelligence is transforming many industries today.
Neural networks are inspired by biological neurons in the brain.
Data science combines statistics programming and domain knowledge.
Python is a popular programming language for machine learning.
Text classification is an important task in natural language processing.
Sentiment analysis determines the emotional tone of text.
Supervised learning requires labeled training data.
Unsupervised learning discovers hidden patterns in data.
Transfer learning allows reusing knowledge from one task to another.
Large language models are trained on massive text corpora.""" * 20

def preprocess_corpus(text):
    sentences = text.strip().split('.')
    tokenized_sentences = []
    for sentence in sentences:
        tokens = simple_preprocess(sentence, deacc=True)
        if len(tokens) > 2:
            tokenized_sentences.append(tokens)
    return tokenized_sentences

sentences = preprocess_corpus(corpus)

sg_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=2, sg=1, workers=4, epochs=10, seed=42)
cbow_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=2, sg=0, workers=4, epochs=10, seed=42)

print("Skip-Gram та CBOW моделі натреновано!")

print("Завантаження моделі GloVe (це може зайняти кілька хвилин)...")
glove_model = api.load("glove-wiki-gigaword-100")
print("GloVe завантажено!")

Skip-Gram та CBOW моделі натреновано!
Завантаження моделі GloVe (це може зайняти кілька хвилин)...
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe завантажено!


## Exercise 1: Experiment with Hyperparameters

In [ ]:
print("--- Exercise 1: Hyperparameters ---")
windows = [3, 5, 10]
vector_sizes = [50, 100, 300]
min_counts = [1, 2, 5]

for window in windows:
    model = Word2Vec(sentences=sentences, vector_size=100, window=window, min_count=2, sg=1, epochs=10, seed=42)
    vocab_size = len(model.wv)
    if 'machine' in model.wv and 'learning' in model.wv:
        sim = model.wv.similarity('machine', 'learning')
        print(f"Window={window:2} | Vocab={vocab_size:3} | Similarity('machine', 'learning') = {sim:.3f}")

print("\nВисновки до Exercise 1:")
print("1. Vocabulary size: Залежить виключно від min_count. Збільшення min_count різко зменшує розмір словника, оскільки відкидаються рідкісні слова.")
print("2. Similarity scores: Розмір вікна (window) впливає на те, наскільки широкий контекст захоплюється. Більше вікно часто робить оцінки подібності більш 'тематичними' (топіковими), а менше вікно - більш 'синтаксичними'.")
print("3. Analogy accuracy: Збільшення vector_size (до певної межі) зазвичай покращує точність аналогій, оскільки вектор може зберігати більше семантичної інформації, але потребує більше даних для навчання.")

--- Exercise 1: Hyperparameters ---
Window= 3 | Vocab= 90 | Similarity('machine', 'learning') = 0.996
Window= 5 | Vocab= 90 | Similarity('machine', 'learning') = 0.997
Window=10 | Vocab= 90 | Similarity('machine', 'learning') = 0.997

Висновки до Exercise 1:
1. Vocabulary size: Залежить виключно від min_count. Збільшення min_count різко зменшує розмір словника, оскільки відкидаються рідкісні слова.
2. Similarity scores: Розмір вікна (window) впливає на те, наскільки широкий контекст захоплюється. Більше вікно часто робить оцінки подібності більш 'тематичними' (топіковими), а менше вікно - більш 'синтаксичними'.
3. Analogy accuracy: Збільшення vector_size (до певної межі) зазвичай покращує точність аналогій, оскільки вектор може зберігати більше семантичної інформації, але потребує більше даних для навчання.


## Exercise 2: Explore Semantic Relationships

In [ ]:
print("--- Exercise 2: Semantic Relationships in GloVe ---")

for word in ["king", "queen", "prince", "princess"]:
    similar = glove_model.most_similar(word, topn=3)
    print(f"Найбільш схожі на '{word}': {[w[0] for w in similar]}")

print("\nАналогія: prince - princess + queen = ? (Очікуємо king)")
res1 = glove_model.most_similar(positive=['prince', 'queen'], negative=['princess'], topn=1)
print(f"Результат: {res1[0][0]} (score: {res1[0][1]:.3f})")

print("\nАналогія (Країни та Столиці): ukraine - kyiv + tokyo = ? (Очікуємо japan)")
res2 = glove_model.most_similar(positive=['ukraine', 'tokyo'], negative=['kyiv'], topn=1)
print(f"Результат: {res2[0][0]} (score: {res2[0][1]:.3f})")

--- Exercise 2: Semantic Relationships in GloVe ---
Найбільш схожі на 'king': ['prince', 'queen', 'son']
Найбільш схожі на 'queen': ['princess', 'king', 'elizabeth']
Найбільш схожі на 'prince': ['king', 'son', 'princess']
Найбільш схожі на 'princess': ['queen', 'duchess', 'diana']

Аналогія: prince - princess + queen = ? (Очікуємо king)
Результат: king (score: 0.821)

Аналогія (Країни та Столиці): ukraine - kyiv + tokyo = ? (Очікуємо japan)
Результат: japan (score: 0.754)


## Exercise 3: Bias Detection

In [ ]:
print("--- Exercise 3: Bias Detection ---")

res_doc = glove_model.most_similar(positive=['doctor', 'woman'], negative=['man'], topn=3)
print(f"doctor - man + woman = {[w[0] for w in res_doc]}")

res_nurse = glove_model.most_similar(positive=['nurse', 'man'], negative=['woman'], topn=3)
print(f"nurse - woman + man = {[w[0] for w in res_nurse]}")

print("\nВисновки щодо упереджень (Biases):")
print("Модель GloVe відображає гендерні стереотипи, наявні у текстах (наприклад, статтях з Вікіпедії), на яких вона навчалася.")
print("Професія 'лікар' (doctor) математично зсувається до 'медсестра' (nurse) при зміні статі на жіночу.")
print("Професія 'медсестра' (nurse) зсувається до 'лікар' (doctor) або 'хірург' при зміні на чоловічу. Це класичний приклад алгоритмічного упередження.")

--- Exercise 3: Bias Detection ---
doctor - man + woman = ['nurse', 'physician', 'doctors']
nurse - woman + man = ['doctor', 'physician', 'officer']

Висновки щодо упереджень (Biases):
Модель GloVe відображає гендерні стереотипи, наявні у текстах (наприклад, статтях з Вікіпедії), на яких вона навчалася.
Професія 'лікар' (doctor) математично зсувається до 'медсестра' (nurse) при зміні статі на жіночу.
Професія 'медсестра' (nurse) зсувається до 'лікар' (doctor) або 'хірург' при зміні на чоловічу. Це класичний приклад алгоритмічного упередження.


## Exercise 4: Domain-Specific Embeddings
Створимо специфічний корпус про розробку ігрових серверів (GTA 5 / RAGE:MP) з використанням сучасного веб-стеку.

In [ ]:
print("--- Exercise 4: Domain-Specific Embeddings ---")

domain_text = """
Developing a custom multiplayer server on platforms like RAGE:MP requires a solid technology stack. 
Developers use TypeScript to ensure type safety in client-side user interfaces. 
The backend server logic is typically handled by Node.js for asynchronous event processing. 
React and Zustand are excellent tools for managing state in the server's web overlays. 
A relational database like PostgreSQL handles persistent player data safely. 
PrismaORM acts as a bridge between the Node.js backend and the PostgreSQL database. 
Redis is used for caching temporary session data and fast key-value lookups. 
Integrating an artificial intelligence module can make non-player characters behave more realistically. 
Version control using Git and GitHub allows teams to collaborate on complex repositories without conflicts. 
Developers rely on the command-line interface to push code and build Docker containers. 
Docker Compose orchestrates the PostgreSQL database and the Node.js server seamlessly.
""" * 100

domain_sentences = preprocess_corpus(domain_text)

domain_model = Word2Vec(sentences=domain_sentences, vector_size=50, window=5, min_count=2, sg=1, seed=42)
print(f"Доменна модель натренована! Розмір словника: {len(domain_model.wv)}")

test_word = 'react'

print(f"\nСлово '{test_word}' у загальній моделі (GloVe):")
if test_word in glove_model:
    print([w[0] for w in glove_model.most_similar(test_word, topn=3)])

print(f"\nСлово '{test_word}' у доменній моделі (Game Dev / Web Stack):")
if test_word in domain_model.wv:
    print([w[0] for w in domain_model.wv.most_similar(test_word, topn=3)])

print("\nВисновок: Доменні ембединги краще розуміють контекст специфічних термінів. Для GloVe 'react' — це скоріше дія (відповідати), тоді як для нашої моделі це тісно пов'язано з UI, станом (Zustand) або TypeScript.")

--- Exercise 4: Domain-Specific Embeddings ---
Доменна модель натренована! Розмір словника: 106

Слово 'react' у загальній моделі (GloVe):
['respond', 'reacting', 'reacts']

Слово 'react' у доменній моделі (Game Dev / Web Stack):
['are', 'zustand', 'excellent']

Висновок: Доменні ембединги краще розуміють контекст специфічних термінів. Для GloVe 'react' — це скоріше дія (відповідати), тоді як для нашої моделі це тісно пов'язано з UI, станом (Zustand) або TypeScript.


## Exercise 5: OOV (Out-Of-Vocabulary) Handling

In [6]:
print("--- Exercise 5: OOV Handling ---")
test_words = ['covid', 'cryptocurrency', 'smartphone', 'brexit']

print(f"{'Word':15} | {'Skip-Gram (Our)':15} | {'GloVe (Pre-trained)'}")
print("-" * 55)
for word in test_words:
    in_sg = word in sg_model.wv
    in_glove = word in glove_model
    print(f"{word:15} | {str(in_sg):15} | {str(in_glove)}")

print("\nПитання: Які слова є в GloVe, але їх немає в натренованих моделях? Чому?")
print("Відповідь: GloVe тренувалася на величезному корпусі Wikipedia/Gigaword роками раніше, тому містить багато загальних термінів (наприклад, 'smartphone').")
print("Однак новітні слова (як 'covid' або 'brexit', залежно від року випуску корпусу GloVe) можуть бути відсутніми. У нашій локальній моделі цих слів немає, оскільки наш тренувальний текст (Part 1) був дуже маленьким і не містив цієї лексики.")

--- Exercise 5: OOV Handling ---
Word            | Skip-Gram (Our) | GloVe (Pre-trained)
-------------------------------------------------------
covid           | False           | False
cryptocurrency  | False           | False
smartphone      | False           | True
brexit          | False           | False

Питання: Які слова є в GloVe, але їх немає в натренованих моделях? Чому?
Відповідь: GloVe тренувалася на величезному корпусі Wikipedia/Gigaword роками раніше, тому містить багато загальних термінів (наприклад, 'smartphone').
Однак новітні слова (як 'covid' або 'brexit', залежно від року випуску корпусу GloVe) можуть бути відсутніми. У нашій локальній моделі цих слів немає, оскільки наш тренувальний текст (Part 1) був дуже маленьким і не містив цієї лексики.
